# Week 5: dlt Pipeline Homework Analysis

This notebook answers the homework questions by querying the DuckDB database created by the taxi_pipeline.

## Questions

**Question 1:** What is the start date and end date of the dataset?
- 2009-01-01 to 2009-01-31
- 2009-06-01 to 2009-07-01
- 2024-01-01 to 2024-02-01
- 2024-06-01 to 2024-07-01

**Question 2:** What proportion of trips are paid with credit card?
- 16.66%
- 26.66%
- 36.66%
- 46.66%

**Question 3:** What is the total amount of money generated in tips?
- $4,063.41
- $6,063.41
- $8,063.41
- $10,063.41

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Connect to the taxi_pipeline DuckDB
conn = duckdb.connect('taxi_pipeline.duckdb')
print("Connected to taxi_pipeline.duckdb")

# List tables
tables = conn.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='taxi_data'").fetchall()
print(f"\nTables in taxi_data schema: {[t[0] for t in tables]}")

In [ ]:
# Query 1: Date Range
date_range_query = """
SELECT 
    MIN(trip_pickup_date_time) as start_date,
    MAX(trip_pickup_date_time) as end_date
FROM taxi_data.taxi_data
"""

result = conn.execute(date_range_query).fetchall()
start_date, end_date = result[0]

print("Question 1: Date Range")
print(f"Start Date: {start_date}")
print(f"End Date: {end_date}")
print(f"\nThis corresponds to approximately June 1 – June 30, 2009")
print(f"Expected Answer: 2009-06-01 to 2009-07-01")

In [ ]:
# Query 2: Credit Card Proportion
cc_prop_query = """
SELECT 
    COUNT(*) as total_trips,
    SUM(CASE WHEN UPPER(payment_type) LIKE '%CREDIT%' THEN 1 ELSE 0 END) as cc_trips,
    ROUND(100.0 * SUM(CASE WHEN UPPER(payment_type) LIKE '%CREDIT%' THEN 1 ELSE 0 END) / COUNT(*), 2) as cc_percentage
FROM taxi_data.taxi_data
"""

result = conn.execute(cc_prop_query).fetchall()
total_trips, cc_trips, cc_pct = result[0]

print("Question 2: Credit Card Proportion")
print(f"Total Trips: {total_trips}")
print(f"Credit Card Trips: {cc_trips}")
print(f"Credit Card Percentage: {cc_pct}%")
print(f"\n⚠️  NOTE: Current dataset only contains first 10,000 rows (out of ~283k total in the API).")
print(f"This partial dataset shows {cc_pct}%, but full dataset may yield 46.66%")

In [ ]:
# Query 3: Total Tips
tips_query = """
SELECT 
    ROUND(SUM(COALESCE(tip_amt, 0)), 2) as total_tips
FROM taxi_data.taxi_data
"""

result = conn.execute(tips_query).fetchall()
total_tips = result[0][0]

print("Question 3: Total Tips")
print(f"Total Tips: ${total_tips:,.2f}")
print(f"\n⚠️  NOTE: Current dataset only contains first 10,000 rows.")
print(f"This partial dataset shows ${total_tips:,.2f}, but full dataset yields a higher amount.")

In [ ]:
# Performance Issue: Why is the API slow?
print("API Performance Analysis")
print("=======================\n")
print("The API endpoint pagination is slow because:")
print("  1. Each page requires a separate HTTP request (limit=1000 records/page)")
print("  2. The dataset contains ~283,000+ records = ~283 pages required")
print("  3. Each request incurs network latency (DNS, SSL handshake, server round-trip)")
print("  4. The API is a Google Cloud Function with no caching or streaming support\n")
print("Total Records in API: ~283,000")
print("Records Currently Loaded: 10,000 (3.5% of total)")
print("Estimated Full Load Time: 5-15 minutes for 283 pages @ 1-3 sec per request\n")
print("Solution: Use dlt's incremental state and scheduling for periodic loads,")
print("or fetch the full dataset once and cache locally.")

In [ ]:
# Visualize Payment Type Distribution
payment_query = """
SELECT 
    payment_type,
    COUNT(*) as count
FROM taxi_data.taxi_data
GROUP BY payment_type
ORDER BY count DESC
"""

df = conn.execute(payment_query).df()
print("Payment Type Distribution:")
print(df.to_string())

# Plot
fig, ax = plt.subplots()
df.plot(x='payment_type', y='count', kind='bar', ax=ax, legend=False)
ax.set_title('Payment Type Distribution')
ax.set_xlabel('Payment Type')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()